# 行业指数顶部与底部信号：净新高占比（(NH-NL)%）
## 背景
### 行业涨跌分化凸显行业指数顶底信号的必要性
一方面，近年来A股行业指数同步涨跌的特征逐渐弱化，特别是2020年之后，行业指数间的分化态势愈发显著，在此背景下，挖掘能够提示行业指数顶部与底部的信号具有重要意义。28个中信一级行业估值分位数的波动率，进一步印证了当前行情的分化特征。另一方面，各类行业指数ETF及主题基金的持续发展与规模扩张，进一步提升了市场对行业指数顶底判断的需求。

### 行业指数顶底信号构建及简单反转策略设计
本文通过构建净新高占比（(NH-NL)%）指标刻画行业指数情绪，该指标延续了对价格新高与价格新低的关注，与我们用于衡量宽基指数情绪的NHNL指标具有一致的设计逻辑，其核心理论依据仍为行为金融学中的锚定效应。基于净新高占比（(NH-NL)%）信号，我们设计了一套包含买入、卖出、仓位管理、止损及移动规则等要素的简单反转策略。

### 净新高占比对科技板块行业指数顶底的提示作用
我们对科技板块中的电子、通信、计算机及传媒行业逐一开展回溯分析，从信号提示准确率、简单反转策略盈利表现等维度进行检验后发现，净新高占比（(NH-NL)%）指标能够有效提示行业指数的顶部与底部；所构建的简单反转策略具备有效性，且2020年以来该策略在科技板块的应用表现较为突出。鉴于反转策略的资金占用周期较短，其更适用于作为指数增强类策略。

近年来，A股行业指数同涨同跌的现象有所减弱，2020年以来行业指数间的分化更为明显；与此同时，随着各行业指数ETF、主题基金的兴起与壮大，市场对行业指数顶部和底部判断的需求持续增加。基于此，我们尝试挖掘可提示行业指数顶底的有效信号，通过构建净新高占比（(NH-NL)%）指标，探索中信一级行业指数的顶底特征。回溯测试结果显示，净新高占比（(NH-NL)%）指标在科技板块与周期板块的表现较为理想。本报告以科技板块为研究案例，复盘了净新高占比（(NH-NL)%）指标提示科技板块行业指数顶底的准确率，以及简单反转策略的实际有效性。

In [9]:
# 基础数据科学计算库导入
# pandas：用于金融时序数据的读取、清洗、合并与结构化处理
# numpy：提供数值计算、数组运算与数学统计基础支持
# 类型注解工具：为函数与变量提供类型标记，提升代码可读性与健壮性
import pandas as pd
import numpy as np
from typing import Dict, List

## 构建逻辑
净新高占比指标$\boldsymbol{(NH-NL)\%}$，为行业范围内年度新高个股数量与年度新低个股数量的差值，占行业内全部个股总量的比例。该指标内在逻辑合理性主要来源于四个维度：

- **锚定效应与价格新高新低**
根据行为金融学相关理论，市场参与者普遍存在显著的锚定效应，投资者会清晰记住自身持仓个股或基金的持仓成本，并将其作为决策锚点，多数交易决策都会依据自身持仓处于浮盈或是浮亏状态进行判断。个股创下年度新高时，持仓投资者整体处于浮盈状态，即便存在卖出意愿，也倾向于等待价格进一步上行，对应个股上方抛压相对有限；而个股创下年度新低时，持仓投资者普遍处于浮亏状态，行情出现反弹便易产生抛售行为，个股短期抛压相对偏大。

- **新高新低直观反映个股运行趋势**
价格新高与价格新低是个股走势强弱最直接的表现形式。长期成长标的往往依托持续抬升的价格新高实现涨幅扩张；而个股深度下行前大多先出现阶段性回调，最先显现的信号便是年度价格新低。

- **新高新低差值体现行业整体个股走势**
行业内年度新高个股数量多于年度新低个股数量，代表行业内多数标的走势偏强；反之则说明行业整体标的走势偏弱，该差值能够有效表征整个行业指数的运行强弱。

- **净新高占比适配行业强弱与市场情绪刻画**
经过归一化处理后指标取值区间为$\boldsymbol{[-1,1]}$，能够剔除不同行业上市标的数量差异带来的干扰（例如机械行业上市满1年标的接近500只，煤炭行业仅36只），同时实现各行业间指标横向对比，弱化行业自身属性带来的偏差。

## 信号构建方法
本文构建净新高占比指标（Net Percent of New High Minus New Low，缩写$\boldsymbol{(NH-NL)\%}$，也记作$\boldsymbol{NPNMN}$），用于衡量中信一级行业指数强弱水平，计算公式如下：
$$净新高占比(NH-NL)\%=\frac{创年度新高的个股数-创年度新低的个股数}{行业内部上市超过1年的个股数}$$

其中年度新高、年度新低定义为：**个股收盘价高于/低于过去52周至前一周区间内的最高/最低收盘价**。

指标界定存在两处特殊设定：
1. 传统判定方式采用最高价、最低价进行极值对比，本指标统一选用收盘价作为判定依据；
2. 历史极值回溯区间选取$T_{1-52}$，并非$T_{0-51}$。

## $\boldsymbol{(NH-NL)\%}$指标的阈值划分标准
针对中信一级行业指数，初始设定阈值：20%、30%对应乐观、贪婪区间，-20%、-30%对应悲观、恐惧区间，区间划分规则如下：
$$NHNL = \begin{cases} 
x \geq 30\% \ 贪婪\\\
20\% \leq x \lt 30\% \ 乐观\\\
-20\% \lt x \lt 20\% \ 正常区间\\\
-30\% \lt x \leq -20\% \ 悲观\\\
x \leq -30\% \ 恐惧
\end{cases}$$

为规避部分行业个股基数偏小造成的指标数值异常波动，若行业内上市满1年标的数量不足40只，则整体放宽阈值，调整为$\pm30\%$、$\pm40\%$，新标准如下：
$$NHNL = \begin{cases} 
x \geq 40\% \ 贪婪\\\
30\% \leq x \lt 40\% \ 乐观\\\
-30\% \lt x \lt 30\% \ 正常区间\\\
-40\% \lt x \leq -30\% \ 悲观\\\
x \leq -40\% \ 恐惧
\end{cases}$$

## 策略跟踪执行规则
基于净新高占比$\boldsymbol{(NH-NL)\%}$构建反转交易策略，具体规则如下：
指标进入贪婪区间后开始纳入监测，当数值首次回落至常规区间时触发做空信号，于下一交易日开盘进场做空。止损位设置为近5个交易日价格最高点，若无触发止损则持仓满30个交易日后以当日收盘价平仓；开仓满一周且未触发止损的情况下，将止损价格调整至开仓成本线。

指标进入恐惧区间后开始纳入监测，当数值首次回升至常规区间时触发做多信号，于下一交易日开盘进场做多。止损位设置为近5个交易日价格最低点，若无触发止损则持仓满30个交易日后以当日收盘价平仓；开仓满一周且未触发止损的情况下，同步将止损价格调整至开仓成本线。

In [68]:
import sys
from pathlib import Path

# 配置项目根目录环境路径
sys.path.append(str(Path().resolve().parents[0]))

# 进度条可视化工具
from tqdm.notebook import tqdm

def calculate_industry_exceed_threshold_count(
    industry_code_mapping: Dict, start_date: str, end_date: str, threshold: int
) -> pd.DataFrame:
    """
    计算行业内股票上市天数超过指定阈值的数量
    计算逻辑：交易日与上市日之间的自然日天数统计
    
    参数说明：
        industry_code_mapping (Dict): 键-股票代码，值-行业名称
        start_date (str): 统计起始日期
        end_date (str): 统计结束日期
        threshold (int): 上市天数阈值
        
    返回结果：
        pd.DataFrame: 索引为交易日，列名为行业名称，值为对应行业超阈值股票数量
    """
    # 加载全市场股票基础信息数据集
    stock_info: pd.DataFrame = get_ts_stock_basic()
    target_stock_codes: List = list(industry_code_mapping.keys())

    # 筛选目标股票池数据
    filter_condition: pd.DataFrame = stock_info["code"].isin(target_stock_codes)
    filtered_stock_info: pd.DataFrame = stock_info[filter_condition]
    
    # 获取指定区间内所有交易日序列
    trading_date_series: np.ndarray = get_trade_days(start_date, end_date)

    # 构建股票代码-上市日期映射关系并转换日期格式
    ipo_date_mapping: pd.Series = filtered_stock_info.set_index("code")["list_date"]
    ipo_date_mapping: pd.Series = ipo_date_mapping.astype(np.datetime64)

    # 构建日期矩阵并广播计算上市天数
    date_matrix: np.ndarray = np.broadcast_to(trading_date_series, (len(target_stock_codes), len(trading_date_series))).T
    ipo_days_calculation: pd.DataFrame = (
        pd.DataFrame(data=date_matrix, index=trading_date_series, columns=target_stock_codes)
        .sub(ipo_date_mapping)
        .applymap(lambda item: item.days)
    )

    # 将股票代码映射为行业名称
    ipo_days_calculation.columns = ipo_days_calculation.columns.map(industry_code_mapping)

    # 按行业分组统计超过阈值的股票数量
    return ipo_days_calculation.apply(lambda series: series > threshold).groupby(level=0, axis=1).sum()

In [7]:
# 定义核心观测日期
monitor_date = "2023-03-03"

# 获取申万一级行业分类数据（指定日期）
industry_category: pd.DataFrame = get_ind_classify(monitor_date, "sw_level1")
# 获取申万一级行业成分股数据（指定日期）
industry_components: pd.DataFrame = get_ind_concept_cons(monitor_date, "sw_level1")

# 构建映射字典：股票代码 -> 行业编码
code_to_industry_id: Dict = industry_components.set_index("code")["industry_code"].to_dict()
# 构建映射字典：行业编码 -> 行业名称
industry_id_to_name: Dict = industry_category.set_index("code")["sec_name"].to_dict()

# 构建最终映射字典：股票代码 -> 行业名称
code_to_industry_name: Dict = {
    stock_code: industry_id_to_name[ind_id]
    for stock_code, ind_id in code_to_industry_id.items()
}

# 可选：统计各行业成分股数量（已注释）
# industry_stock_count: pd.Series = pd.Series(Counter(code_to_industry_name.values()))

In [70]:
# 查看申万一级行业分类数据前5行，快速校验数据结构与字段
industry_category.head()

,code,trade_date,sec_name
0,801010.SI,2023-03-03,农林牧渔
1,801030.SI,2023-03-03,基础化工
2,801040.SI,2023-03-03,钢铁
3,801050.SI,2023-03-03,有色金属
4,801080.SI,2023-03-03,电子


In [71]:
# 查看申万一级行业成分股数据前5行，校验数据结构与内容格式
industry_components.head()

,code,trade_date,sec_name,industry_code
0,000019.SZ,2023-03-03,深粮控股,801010.SI
1,000505.SZ,2023-03-03,京粮控股,801010.SI
2,000592.SZ,2023-03-03,平潭发展,801010.SI
3,000639.SZ,2023-03-03,西王食品,801010.SI
4,000663.SZ,2023-03-03,永安林业,801010.SI


In [8]:
# 获取申万一级行业指数的行情数据
industry_price_data: pd.DataFrame = get_ind_price(
    list(industry_id_to_name.keys()),
    start_date="2014-01-01",
    end_date="2023-03-03",
    fields=["close"],
    level="sw_level1",
)

# 将行业代码映射为行业名称，便于后续阅读
industry_price_data["name"] = industry_price_data["code"].map(industry_id_to_name)

# 数据透视：构建日期为索引、行业名称为列、收盘价为值的数据表
pivoted_industry_price: pd.DataFrame = pd.pivot_table(
    industry_price_data, index="trade_date", columns="name", values="close"
)

In [72]:
# 查看行业价格基础数据集前5行，验证数据结构与字段信息
industry_price_data.head()

,code,close,trade_date,name
0,801010.SI,3265.37,2022-11-16,农林牧渔
1,801030.SI,4242.14,2022-11-16,基础化工
2,801040.SI,2350.87,2022-11-16,钢铁
3,801050.SI,5028.19,2022-11-16,有色金属
4,801080.SI,3808.33,2022-11-16,电子


In [74]:
# ================================
# 批量获取2013-2023年股票行情数据
# 数据源：申万一级行业成分股 | 字段：最高价、最低价 | 复权方式：后复权
# ================================

# 提取行业内所有唯一股票代码列表
stock_code_list: List = industry_components["code"].unique().tolist()

# 构建年度起始日期序列（每年1月1日）
start_date_series = pd.date_range(
    "2013-01-01", 
    monitor_date, 
    freq="YS"
)

# 构建年度结束日期序列，并追加最终观测日期
end_date_series = (
    pd.date_range("2013-01-01", monitor_date, freq="Y")
    .append(pd.to_datetime([monitor_date]))
    .unique()
)

# 循环批量拉取行情数据（带进度条）
data_frame_collector: List = [
    get_ts_price(
        stock_code_list,
        start_date=start_pt.strftime("%Y-%m-%d"),
        end_date=end_pt.strftime("%Y-%m-%d"),
        fields=["high", "low"],
        fq="post",
    )
    for start_pt, end_pt in tqdm(
        zip(start_date_series, end_date_series),
        total=len(start_date_series),
        desc="数据查询进度"
    )
]

# 合并所有时间段的行情数据
merged_price_data: pd.DataFrame = pd.concat(data_frame_collector)

数据查询:   0%|          | 0/11 [00:00<?, ?it/s]

In [76]:
# 查看合并后的股票价格数据集前5行，校验数据结构与字段信息
merged_price_data.head()

,trade_date,code,high,low
0,2013-01-04,600839.SH,56.35894,55.28030
1532,2013-01-04,300271.SZ,24.45246,23.45694
1533,2013-01-04,002678.SZ,11.80000,11.50000
1534,2013-01-04,002394.SZ,24.86005,24.03910
1535,2013-01-04,300323.SZ,11.83000,11.28000


In [10]:
# 构建数据透视表：以交易日为索引，股票代码为列，展示最高价与最低价
reshaped_price_table: pd.DataFrame = pd.pivot_table(
    merged_price_data,
    index="trade_date",
    columns="code",
    values=["low", "high"]
)

In [16]:
# 提取数据的时间范围：起始日期与结束日期
analysis_start_date = reshaped_price_table.index.min()
analysis_end_date = reshaped_price_table.index.max()

# 计算行业内上市天数超过阈值（252个交易日）的股票数量
industry_count_result: pd.DataFrame = calculate_industry_exceed_threshold_count(
    code_to_industry_name, analysis_start_date, analysis_end_date, 252
)

In [6]:
# 查看行业上市天数超阈值统计结果的前5行，校验数据格式
industry_count_result.head()

,交通运输,传媒,公用事业,农林牧渔,医药生物,商贸零售,国防军工,基础化工,家用电器,建筑材料,...,纺织服饰,综合,美容护理,计算机,轻工制造,通信,钢铁,银行,非银金融,食品饮料
2013-01-04,82,78,82,63,191,78,57,159,38,53,...,53,24,8,123,47,50,36,16,55,60
2013-01-07,82,79,82,63,191,78,57,159,38,53,...,53,24,8,123,47,50,36,16,55,60
2013-01-08,82,79,82,63,191,78,57,159,38,53,...,53,24,8,123,47,50,36,16,55,60
2013-01-09,82,79,82,63,191,78,57,159,38,53,...,53,24,8,123,47,50,36,16,55,60
2013-01-10,82,79,82,63,191,78,57,159,38,53,...,53,24,8,123,47,50,36,16,55,60


### 信号构建
本次依托现有存量数据直接完成对应交易信号的搭建工作。

此处存在一处需要留意的细节：中信一级行业成分股每间隔半年进行一次调整，本次信号计算仅选取最新截面下的行业成分股进行生成，并未结合历史成分股权重动态回溯，因此信号对应的前期历史行情数据可能存在一定程度失真。

In [10]:
# 数据获取核心模块
from source import get_data

# 行业指标计算核心模块
from source import calc_industry_nhnl

# 可视化绘图工具函数
from source import plot_nhnl_signal

# Plotly 交互式绘图环境配置（Notebook 专用）
from plotly.offline import init_notebook_mode, iplot

# 初始化 Plotly 笔记本绘图模式
init_notebook_mode(True)

In [11]:
# 计算行业新高新低指标（NH-NL）
# 参数说明：价格透视表 | 行业名称映射 | 计算周期(52*5) | 上市天数统计 | 非传统计算模式
nhnl_indicator_data: pd.DataFrame = calc_industry_nhnl(
    reshaped_price_table,
    code_to_industry_name,
    52 * 5,
    industry_count_result,
    tradition=False
)

In [12]:
# 生成电子行业净新高信号可视化图表
# 参数：行业价格序列 | 行业NHNL指标 | 信号阈值 | 图表标题 | 开启绘图配置
visual_chart = plot_nhnl_signal(
    pivoted_industry_price["电子"],
    nhnl_indicator_data["电子"],
    40,
    "电子行业净新高占比",
    True
)

# 在 Notebook 中渲染交互式图表
iplot(visual_chart)